# install dependencies

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
import statsmodels.api as sm
import pyreadstat

FEATURE_PREFIXES = ('Work Activities_', 'Skills_', 'Knowledge_')

def clean_feature_name(name):
    name = str(name)
    for prefix in FEATURE_PREFIXES:
        if name.startswith(prefix):
            return name[len(prefix):]
    return name

def clean_feature_names(names):
    return [clean_feature_name(name) for name in names]


# Phase1 Data Preparation

In [2]:
def calculate_cir_series(
    outcome,
    file_name,
    data_path='../../result/occ/analysis',
    horizon_start=1,
    horizon_end=36,
    series_name=None
):
    file_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df = pd.read_csv(file_path)

    if series_name is None:
        series_name = f"CIR_{horizon_end}"

    df_outcome = df[df['outcome'] == outcome].copy()
    df_outcome['horizon'] = pd.to_numeric(df_outcome['horizon'])
    df_outcome = df_outcome.sort_values('horizon').reset_index(drop=True)

    group_cols = [col for col in df_outcome.columns if col.startswith('group')]

    cir_dict = {}

    for col in group_cols:
        irf_window = df_outcome[
            df_outcome['horizon'].between(horizon_start, horizon_end)
        ][col]

        cir_dict[col] = irf_window.sum()

    cir_series = pd.Series(cir_dict, name=series_name)
    return cir_series

In [3]:
def build_y_series_from_mapping(
    cir_series,
    file_name,
    data_path='../../result/mapping',
    sheet_name='Sheet1',
    usecols='A,E,F,H',
    series_name=None
):
    mapping_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df_map = pd.read_excel(mapping_path, sheet_name=sheet_name, usecols=usecols, header=0)
    df_map.columns = ['occ1990', 'SOC-2018', 'Group', 'Weights']

    if series_name is None:
        series_name = cir_series.name if cir_series.name is not None else 'value'

    df_map['occ1990'] = pd.to_numeric(df_map['occ1990'], errors='coerce').astype('Int64')
    df_map['SOC-2018'] = df_map['SOC-2018'].astype(str).str.strip()
    df_map = df_map.dropna(subset=['occ1990', 'SOC-2018', 'Group'])

    def occ1990_to_group(occ):
        if 3 <= occ <= 37: return 1
        elif 43 <= occ <= 200: return 2
        elif 203 <= occ <= 235: return 3
        elif 243 <= occ <= 283: return 4
        elif 303 <= occ <= 389: return 5
        elif 405 <= occ <= 469: return 6
        elif (473 <= occ <= 498) or (558 <= occ <= 599) or (614 <= occ <= 617): return 7
        elif (503 <= occ <= 549) or (628 <= occ <= 699): return 8
        elif (703 <= occ <= 799) or (803 <= occ <= 889): return 9
        return np.nan

    df_map['group'] = df_map['occ1990'].apply(occ1990_to_group)

    group_to_value = {}
    for idx, val in cir_series.items():
        match = re.search(r'group(\d+)', str(idx))
        if match:
            group_to_value[int(match.group(1))] = val

    # 1. 先映射 group shock
    df_map['cir_value'] = df_map['group'].map(group_to_value)
    df_map = df_map.dropna(subset=['cir_value'])

    df_final = df_map.drop_duplicates(subset='SOC-2018', keep='first')
    y_series = df_final.set_index('SOC-2018')['cir_value'].dropna()

    # 同时返回权重
    w_series = df_final.set_index('SOC-2018')['Weights'].dropna()

    y_series = (y_series - y_series.mean()) / y_series.std()

    return y_series, w_series

In [4]:
def load_and_prepare_onet_data_extended(
    y_series,
    file_names,
    mapping_path,
    onet_data_path='../../data/ONET',
    mapping_sheet='Sheet1',
    scale_id='IM',
    usecols=[0, 1, 4, 5, 7]
):

    # ── 1. 读取 mapping：A=occ1990, B=occ1990dd, E=SOC_2018 ─────────────
    df_map = pd.read_excel(
        mapping_path,
        sheet_name=mapping_sheet,
        usecols='A,B,E',
        header=0
    )

    df_map.columns = ['occ1990', 'occ1990dd', 'SOC-2018']

    df_map['occ1990'] = pd.to_numeric(
        df_map['occ1990'],
        errors='coerce'
    ).astype('Int64')

    df_map['SOC-2018'] = (
        df_map['SOC-2018']
        .astype(str)
        .str.strip()
    )

    valid_soc = set(df_map['SOC-2018'].unique())
    print(f"mapping 中有效 SOC-2018 数量: {len(valid_soc)}")

    # ── 2. 读取四个 O*NET 文件，只保留 valid_soc ──────────────────────
    dfs = []

    for prefix, fname in file_names.items():
        fpath = f"{onet_data_path.rstrip('/\\\\')}/{fname}"

        df = pd.read_excel(fpath, usecols=usecols, header=0)
        df.columns = [
            'SOC_Code',
            'Sub_Code',
            'Element_Name',
            'Scale_ID',
            'Data_Value'
        ]

        df['SOC_Code'] = df['SOC_Code'].astype(str).str.strip()
        df['Element_Name'] = df['Element_Name'].astype(str).str.strip()
        df['Scale_ID'] = df['Scale_ID'].astype(str).str.strip().str.upper()
        df['Sub_Code'] = df['Sub_Code'].astype(str).str.strip().str.zfill(2)

        means = (
            df.groupby(['SOC_Code', 'Element_Name'])['Data_Value']
            .mean()
            .reset_index()
        )
        means.rename(columns={'Data_Value': 'Mean_Val'}, inplace=True)

        df = df.merge(
            means,
            on=['SOC_Code', 'Element_Name'],
            how='left'
        )

        df.loc[
            df['Sub_Code'] == '00',
            'Data_Value'
        ] = df.loc[
            df['Sub_Code'] == '00',
            'Mean_Val'
        ]

        df = df[df['Sub_Code'] == '00'].copy()
        df = df[df['Scale_ID'] == scale_id].copy()
        df = df.dropna(subset=['Data_Value'])

        df.drop(
            columns=['Mean_Val', 'Sub_Code', 'Scale_ID'],
            inplace=True
        )

        # 直接按 SOC_Code 与 mapping 的 SOC_2018 匹配
        df = df[df['SOC_Code'].isin(valid_soc)].copy()

        df['Element_Name'] = prefix + '_' + df['Element_Name']

        dfs.append(df)

    # ── 3. 合并 O*NET wide format ─────────────────────────────
    df_all = pd.concat(dfs, ignore_index=True)

    df_wide = df_all.pivot_table(
        index='SOC_Code',
        columns='Element_Name',
        values='Data_Value',
        aggfunc='mean'
    ).astype(float)

    df_wide = df_wide.fillna(df_wide.median())

    print(
        f"O*NET 合并后: {df_wide.shape[0]} 个 SOC, "
        f"{df_wide.shape[1]} 个特征"
    )

    # ── 4. 标准化 ─────────────────────────────────────────────
    scaler = StandardScaler()

    X_df = pd.DataFrame(
        scaler.fit_transform(df_wide),
        columns=df_wide.columns,
        index=df_wide.index
    )

    # ── 5. 和 y_series 对齐 ───────────────────────────────────
    aligned_idx = X_df.index.intersection(y_series.index)

    X = X_df.loc[aligned_idx].values
    y_aligned = y_series.loc[aligned_idx].values

    print(
        f"X shape: {X.shape} | "
        f"Aligned samples: {len(aligned_idx)}"
    )

    return X_df, aligned_idx, X, y_aligned

# Phase2: LASSO Estimation

In [5]:
def run_lasso_selection(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_splits=10,
    random_state=42,
    max_iter=5000,
    sample_weight=None,
    use_1se=True  # ← 新增，默认开启
):
    feature_names = pd.Index(feature_names)
    cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    lasso_cv = LassoCV(
        alphas=None,
        cv=cv_strategy,
        max_iter=max_iter,
        random_state=random_state,
        n_jobs=-1
    )
    lasso_cv.fit(X, y_aligned, sample_weight=sample_weight)

    # ── 1-SE 规则 ──────────────────────────────────────────────
    if use_1se:
        mean_mse = lasso_cv.mse_path_.mean(axis=1)  # shape: (n_alphas,)
        std_mse = lasso_cv.mse_path_.std(axis=1) / np.sqrt(n_splits)

        best_idx = np.argmin(mean_mse)
        threshold = mean_mse[best_idx] + std_mse[best_idx]

        # alphas_ 是降序排列，valid 里取最大的 alpha（最稀疏）
        valid_mask = mean_mse <= threshold
        alpha_1se = lasso_cv.alphas_[valid_mask][0]

        # 用 1-SE alpha 重新fit一次得到系数
        from sklearn.linear_model import Lasso
        lasso_final = Lasso(alpha=alpha_1se, max_iter=max_iter)
        lasso_final.fit(X, y_aligned, sample_weight=sample_weight)
        best_alpha = alpha_1se
        best_coefs = lasso_final.coef_

        print(f"CV best alpha: {lasso_cv.alpha_:.6f} → 1-SE alpha: {alpha_1se:.6f}")
    else:
        best_alpha = lasso_cv.alpha_
        best_coefs = lasso_cv.coef_
    # ───────────────────────────────────────────────────────────

    cv_mse_path = lasso_cv.mse_path_
    cv_mean_mse = lasso_cv.mse_path_.mean(axis=1)

    nonzero_mask = best_coefs != 0
    nonzero_idx = np.where(nonzero_mask)[0]
    n_nonzero = len(nonzero_idx)

    nonzero_idx_sorted = nonzero_idx[np.argsort(np.abs(best_coefs[nonzero_idx]))[::-1]]
    top_idx = nonzero_idx_sorted[:top_n]

    top_names_raw = feature_names.take(top_idx).tolist()
    top_names = clean_feature_names(top_names_raw)
    top_coefs = best_coefs[top_idx]
    top_feature_pairs = [f"{name} ({coef:.6f})" for name, coef in zip(top_names, top_coefs)]

    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[top_idx] = True

    if n_nonzero < top_n:
        print(f"LASSO only selects {n_nonzero} non-zero variables, fewer than top_n={top_n}, actually using {n_nonzero} variables")

    print(f"Alpha: {best_alpha:.6f} | Non-zero coefs: {n_nonzero} / {len(feature_names)}")

    return {
        'lasso_cv': lasso_cv,
        'best_alpha': best_alpha,
        'best_coefs': best_coefs,
        'cv_mse_path': cv_mse_path,
        'cv_mean_mse': cv_mean_mse,
        'feature_names': feature_names,
        'top_idx': top_idx,
        'top_names_raw': top_names_raw,
        'top_names': top_names,
        'top_coefs': top_coefs,
        'top_feature_pairs': top_feature_pairs,
        'selected_mask': selected_mask
    }

In [6]:
def lasso_stability_check(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_boots=100,
    n_splits=10,
    freq_threshold=0.9,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    rng = np.random.default_rng(random_state)
    n = len(y_aligned)
    selection_counts = np.zeros(len(feature_names))

    for i in range(n_boots):
        idx = rng.integers(0, n, size=n)
        X_b, y_b = X[idx], y_aligned[idx]
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=int(rng.integers(9999)))
        m = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1).fit(X_b, y_b)
        selection_counts += (m.coef_ != 0).astype(int)

    freq = pd.Series(selection_counts / n_boots, index=feature_names)
    freq = freq.sort_values(ascending=False)

    # 稳定变量：频率 >= freq_threshold
    stable_features = freq[freq >= freq_threshold].index.tolist()
    # 在稳定变量里再截断到 top_n
    final_features = stable_features[:top_n]

    print(f"=== Bootstrap 稳定性检验 (n_boots={n_boots}, threshold={freq_threshold}) ===")
    print(f"频率 >= {freq_threshold} 的变量: {len(stable_features)} 个")
    print(f"频率 >= 0.8 的变量 (高稳定): {(freq >= 0.8).sum()} 个")
    print(f"最终进入 OLS 的变量: {len(final_features)} 个\n")
    print("选中频率 top 15:")
    print(freq.head(15).round(3).to_string())

    # 构建 selected_mask（基于稳定变量，而非单次 LASSO）
    selected_mask = np.zeros(len(feature_names), dtype=bool)
    for name in final_features:
        selected_mask[feature_names.get_loc(name)] = True

    return {
        'freq': freq,
        'stable_features': stable_features,
        'final_features': final_features,
        'selected_mask': selected_mask
    }

In [7]:
def calculate_post_lasso_r2(X, y_aligned, selected_mask, selected_feature_names, sample_weight=None):
    X_selected = X[:, selected_mask]
    X_selected_const = sm.add_constant(X_selected)
    selected_feature_names = clean_feature_names(selected_feature_names)

    ols_model = sm.WLS(y_aligned, X_selected_const, weights=sample_weight).fit(cov_type='HC3')
    r_squared = ols_model.rsquared
    r_squared_adj = ols_model.rsquared_adj
    ci = ols_model.conf_int()

    ols_results_df = pd.DataFrame({
        'O*NET_Activity': selected_feature_names,
        'OLS_Coefficient': ols_model.params[1:],
        'CI_Lower': ci[1:, 0],
        'CI_Upper': ci[1:, 1],
        'Std_Error': ols_model.bse[1:],
        'P_Value': ols_model.pvalues[1:],
        'Sig_10%': ols_model.pvalues[1:] < 0.10,
        'Sig_5%': ols_model.pvalues[1:] < 0.05
    }).sort_values('OLS_Coefficient', key=abs, ascending=False)

    return {
        'ols_model': ols_model,
        'r_squared': r_squared,
        'r_squared_adj': r_squared_adj,
        'ols_results_df': ols_results_df
    }

In [8]:
def run_elasticnet_selection(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_splits=10,
    random_state=42,
    max_iter=5000,
    sample_weight=None,
    use_1se=True
):
    from sklearn.linear_model import ElasticNetCV, ElasticNet

    feature_names = pd.Index(feature_names)
    cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    enet_cv = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        alphas=None,
        cv=cv_strategy,
        max_iter=max_iter,
        random_state=random_state,
        n_jobs=-1
    )
    enet_cv.fit(X, y_aligned, sample_weight=sample_weight)

    best_l1_ratio = enet_cv.l1_ratio_

    # ── 1-SE 规则 ──────────────────────────────────────────────
    if use_1se:
        # mse_path_ shape: (n_l1_ratio, n_alphas, n_folds)
        # 找到最优 l1_ratio 对应的 index
        l1_ratios = np.array(enet_cv.l1_ratio) if hasattr(enet_cv, 'l1_ratio') else np.array([0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0])
        best_l1_idx = np.where(l1_ratios == best_l1_ratio)[0][0]

        # 取该 l1_ratio 下的 mse path
        mse_path_best = enet_cv.mse_path_[best_l1_idx]  # shape: (n_alphas, n_folds)
        mean_mse = mse_path_best.mean(axis=1)
        std_mse = mse_path_best.std(axis=1) / np.sqrt(n_splits)

        best_idx = np.argmin(mean_mse)
        threshold = mean_mse[best_idx] + std_mse[best_idx]

        valid_mask = mean_mse <= threshold
        alphas_best = enet_cv.alphas_[best_l1_idx]  # shape: (n_alphas,)
        alpha_1se = alphas_best[valid_mask][0]  # 降序，取第一个即最大

        enet_final = ElasticNet(
            alpha=alpha_1se,
            l1_ratio=best_l1_ratio,
            max_iter=max_iter
        )
        enet_final.fit(X, y_aligned, sample_weight=sample_weight)
        best_alpha = alpha_1se
        best_coefs = enet_final.coef_

        print(f"CV best alpha: {enet_cv.alpha_:.6f}, l1_ratio: {best_l1_ratio:.2f} → 1-SE alpha: {alpha_1se:.6f}")
    else:
        best_alpha = enet_cv.alpha_
        best_coefs = enet_cv.coef_
    # ───────────────────────────────────────────────────────────

    nonzero_mask = best_coefs != 0
    nonzero_idx = np.where(nonzero_mask)[0]
    n_nonzero = len(nonzero_idx)

    nonzero_idx_sorted = nonzero_idx[np.argsort(np.abs(best_coefs[nonzero_idx]))[::-1]]
    top_idx = nonzero_idx_sorted[:top_n]

    top_names_raw = feature_names.take(top_idx).tolist()
    top_names = clean_feature_names(top_names_raw)
    top_coefs = best_coefs[top_idx]
    top_feature_pairs = [f"{name} ({coef:.6f})" for name, coef in zip(top_names, top_coefs)]

    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[top_idx] = True

    if n_nonzero < top_n:
        print(f"ElasticNet only selects {n_nonzero} non-zero variables, fewer than top_n={top_n}, actually using {n_nonzero} variables")

    print(f"Alpha: {best_alpha:.6f} | l1_ratio: {best_l1_ratio:.2f} | Non-zero coefs: {n_nonzero} / {len(feature_names)}")

    return {
        'enet_cv': enet_cv,
        'best_alpha': best_alpha,
        'best_l1_ratio': best_l1_ratio,
        'best_coefs': best_coefs,
        'feature_names': feature_names,
        'top_idx': top_idx,
        'top_names_raw': top_names_raw,
        'top_names': top_names,
        'top_coefs': top_coefs,
        'top_feature_pairs': top_feature_pairs,
        'selected_mask': selected_mask
    }

# Main

In [9]:
def main():
    # ----------------------------
    # file settings
    # ----------------------------
    irf_file = "merged_occ_irf_trajectories.csv"
    mapping_file = "mapping_done.xlsx"

    file_sets = {
        "Work Activities": {"Work Activities": "Work Activities.xlsx"},
        "Knowledge":        {"Knowledge":        "Knowledge.xlsx"},
        "Skills":           {"Skills":           "Skills.xlsx"}
    }

    outcomes = [
        'unemployment', 'employment', 'income', 'hourly_rate',
        'hours', 'income_share', 'inequality', 'median'
    ]

    all_results = []

    # ----------------------------
    # loop outcomes
    # ----------------------------
    for outcome in outcomes:

        print("=" * 80)
        print(f"Outcome = {outcome}")
        print("=" * 80)

        # Step 1: build y = CIR(1~36)
        cir_series = calculate_cir_series(
            outcome=outcome,
            file_name=irf_file,
            horizon_start=1,
            horizon_end=36,
            series_name=f"{outcome}_CIR36"
        )

        y_series, w_series = build_y_series_from_mapping(
            cir_series=cir_series,
            file_name=mapping_file,
            series_name=outcome
        )

        # ----------------------------
        # loop tables
        # ----------------------------
        for table_name, file_dict in file_sets.items():

            print("-" * 80)
            print(f"{outcome} | {table_name}")
            print("-" * 80)

            # Step 2: X matrix
            X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
                y_series=y_series,
                file_names=file_dict,
                mapping_path=f"../../result/mapping/{mapping_file}"
            )

            w_aligned = w_series.loc[aligned_idx].values

            # Step 3: LASSO + ElasticNet
            lasso_res = run_lasso_selection(
                X=X, y_aligned=y_aligned,
                feature_names=X_df.columns,
                top_n=10, sample_weight=w_aligned
            )

            enet_res = run_elasticnet_selection(
                X=X, y_aligned=y_aligned,
                feature_names=X_df.columns,
                top_n=10, sample_weight=w_aligned
            )

            # ----------------------------
            # loop methods
            # ----------------------------
            for method, res in [("LASSO", lasso_res), ("ElasticNet", enet_res)]:

                selected_mask = res['selected_mask']
                selected_feature_names = pd.Index(clean_feature_names(X_df.columns[selected_mask]))
                l1_ratio = res.get('best_l1_ratio', np.nan)

                if selected_mask.sum() == 0:
                    print(f"{method}: No variable selected")
                    all_results.append({
                        "outcome":       outcome,
                        "table":         table_name,
                        "method":        method,
                        "r_squared":     np.nan,
                        "r_squared_adj": np.nan,
                        "alpha":         res['best_alpha'],
                        "l1_ratio":      l1_ratio,
                        "n_selected":    0,
                        "top10_features": "",
                        "top10_coefs": "",
                        "top10_features_with_coef": ""
                    })
                    continue

                post_res = calculate_post_lasso_r2(
                    X=X, y_aligned=y_aligned,
                    selected_mask=selected_mask,
                    selected_feature_names=selected_feature_names,
                    sample_weight=w_aligned
                )

                print(f"[{method}] R²={post_res['r_squared']:.4f} | Adj R²={post_res['r_squared_adj']:.4f}")
                print(f"Top variables:")
                for i, (name, coef) in enumerate(zip(res['top_names'], res['top_coefs']), 1):
                    print(f"  {i}. {name}: {coef:.6f}")

                all_results.append({
                    "outcome":        outcome,
                    "table":          table_name,
                    "method":         method,
                    "r_squared":      post_res['r_squared'],
                    "r_squared_adj":  post_res['r_squared_adj'],
                    "alpha":          res['best_alpha'],
                    "l1_ratio":       l1_ratio,
                    "n_selected":     selected_mask.sum(),
                    "top10_features": " | ".join(res['top_names']),
                    "top10_coefs":    " | ".join([f"{coef:.6f}" for coef in res['top_coefs']]),
                    "top10_features_with_coef": " | ".join(res['top_feature_pairs'])
                })

    # ----------------------------
    # final summary
    # ----------------------------
    result_df = pd.DataFrame(all_results)

    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    print(result_df[['outcome', 'table', 'method', 'r_squared', 'r_squared_adj',
                      'alpha', 'l1_ratio', 'n_selected']].to_string())

    result_df.to_csv("../../result/occ/analysis/lasso_enet_results.csv", index=False)
    print("\nSaved: lasso_enet_results.csv")


# %%
if __name__ == "__main__":
    main()

Outcome = unemployment
--------------------------------------------------------------------------------
unemployment | Work Activities
--------------------------------------------------------------------------------
mapping 中有效 SOC-2018 数量: 663
O*NET 合并后: 595 个 SOC, 41 个特征
X shape: (595, 41) | Aligned samples: 595
CV best alpha: 0.012906 → 1-SE alpha: 0.148383
LASSO only selects 7 non-zero variables, fewer than top_n=10, actually using 7 variables
Alpha: 0.148383 | Non-zero coefs: 7 / 41
CV best alpha: 0.040119, l1_ratio: 0.30 → 1-SE alpha: 0.461274
ElasticNet only selects 9 non-zero variables, fewer than top_n=10, actually using 9 variables
Alpha: 0.461274 | l1_ratio: 0.30 | Non-zero coefs: 9 / 41
[LASSO] R²=0.4862 | Adj R²=0.4801
Top variables:
  1. Performing for or Working Directly with the Public: -0.230205
  2. Organizing, Planning, and Prioritizing Work: 0.221777
  3. Updating and Using Relevant Knowledge: 0.172978
  4. Processing Information: 0.081686
  5. Selling or Influencin